# GNN + Sparsity-Matched Connectome-Like Network

**Ablation:** only **sparsity** matches connectome (mask IoU << 1).

Artifacts: `/tmp/gnn_bpu_notebook_random_sparsity/`. 


In [1]:
import tempfile
from pathlib import Path

# --- edit training config here ---
TRAIN_SIZES = [530, 5300]
SEED = 1
DEVICE = None          # None = auto
SKIP_IF_EXISTS = True  # skip run if model.pth already exists

# --- weight value range (random inits only; droso unchanged) ---
MATCH_CONNECTOME_WEIGHT_RANGE = False  # True -> match droso block mean/std
CUSTOM_WEIGHT_SCALE = None             # e.g. 0.5; overrides default when set
DEFAULT_WEIGHT_SCALE = 100.0             # std scale when both above are off


# runs/logs go to custom dir; reuse chess subsamples from train_DPU.ipynb
_ORIG_TMP = Path(tempfile.gettempdir()) / "gnn_bpu_notebook"
NOTEBOOK_TMP = Path(tempfile.gettempdir()) / "gnn_bpu_notebook_random_sparsity"

SAVE_CONNECTOME_WEIGHTS_CSV = True
CONNECTOME_WEIGHTS_DIR = NOTEBOOK_TMP / "connectome_weights"
EXP_RUN_DIR = str(NOTEBOOK_TMP / "runs")
SUBSAMPLE_DIR = _ORIG_TMP / "subsamples"
TRAIN_BAG_DIR = _ORIG_TMP / "chess_bags"
CHESS_DATA_ROOT = _ORIG_TMP / "chess_data"

_cwd = Path.cwd()
_GNN_BPU_ROOT = (_cwd / "gnn_bpu") if (_cwd / "gnn_bpu").is_dir() else _cwd
DROSO_DATA_ROOT = _GNN_BPU_ROOT / "droso_data"

BASE_CONFIG = {
    "result_path": "results",
    "seed": SEED,
    "data_choice": "chess_SV",
    "sampling": True,
    "signed": True,
    "conn_type": "ad",
    "input_type": "sensory",
    "output_type": "output",
    "trainable_sparse": False,       # False -> connectome-like weights frozen during training
    "sparse_type": "input_output",
    "connectome_type": "2layer",
    "init_type": "random_sparsity",  # random mask, matched nnz/sparsity only
    "match_connectome_weight_range": MATCH_CONNECTOME_WEIGHT_RANGE,
    "custom_weight_scale": CUSTOM_WEIGHT_SCALE,
    "default_weight_scale": DEFAULT_WEIGHT_SCALE,
    "save_connectome_weights_csv": SAVE_CONNECTOME_WEIGHTS_CSV,
    "connectome_weights_csv_path": str(CONNECTOME_WEIGHTS_DIR / f"init_seed{SEED}.csv"),
    "numeric_input": True,
    "GNN_only": True,
    "GCNN": False,
    "use_steps": True,
    "num_steps": 400_000,
    "num_steps_eval": 10_000,
    "puzzle_steps_eval": 20_000,
    "learning_rate": 1e-4,
    "dropout_rate": 0.2,
    "use_cos_schedule": False,
    "init": "droso",
    "learnable": False,
    "model_choice": {
        "GINE": {"hidden_dim": 128, "num_layer": 2, "residual": False, "gated": False},
        "CNN3L": {"filter_num": 128, "feat_dim": 256, "reduction_ratio": 16, "drop_out": True, "dropout_rate": 0.2},
    },
}

DROSO_CONFIG = {
    "droso_dir": str(DROSO_DATA_ROOT),
    "annotation_path": str(DROSO_DATA_ROOT / "neuron_category.pkl"),
    "sio": True,
    "rescale_factor": 0.04,
}

In [ ]:
import copy
import contextlib
import os
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path
from urllib.request import urlretrieve

import torch
from torch_geometric.loader import DataLoader

from src import chess_config as config_lib
from src.basics import get_exp_name
from src.chess_utils import data_loader
from src.chess_utils.data_loader import (
    ChessDataset,
    NpyChessDataset,
    _TRANSFORMATION_BY_POLICY,
    save_dataset_to_npy,
    seed_worker,
)
from src.evaluate.evaluate_DPU_on_puzzles import eval_DPU_on_puzzles
from src.evaluate.plotting import plot_loss_curve
from src.train_utils import training
from src.train_utils.training import get_out_path

# project root
PROJECT_ROOT = os.getcwd()
if os.path.isdir(os.path.join(os.getcwd(), "gnn_bpu")):
    PROJECT_ROOT = os.path.join(os.getcwd(), "gnn_bpu")
os.chdir(PROJECT_ROOT)

EVAL_DATA_ROOT = CHESS_DATA_ROOT
CHESS_DATA_URL = "https://storage.googleapis.com/searchless_chess/data"
SUBSAMPLE_NPROCS = 20

NOTEBOOK_TMP.mkdir(parents=True, exist_ok=True)
SUBSAMPLE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_BAG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = NOTEBOOK_TMP / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)


@contextlib.contextmanager
def log_to_file(log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "w", encoding="utf-8") as f:
        with redirect_stdout(f), redirect_stderr(f):
            yield log_path


def resolve_device(device=None):
    if device is None:
        if torch.cuda.is_available():
            device = "cuda:0"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"
    elif device == "cuda":
        device = "cuda:0"
    if device.startswith("cuda"):
        torch.cuda.set_device(int(device.split(":")[1]))
    return device


def subsample_npy_path(num_records, seed):
    return SUBSAMPLE_DIR / f"subsample_{num_records}_seed{seed}.npy"


REQUIRED_DROSO_FILES = [
    DROSO_DATA_ROOT / "neuron_category.pkl",
    DROSO_DATA_ROOT / "ad_signed_connectivity_matrix.csv",
]


def ensure_droso_data():
    missing = [p for p in REQUIRED_DROSO_FILES if not p.exists()]
    if missing:
        names = ", ".join(p.name for p in missing)
        raise FileNotFoundError(
            f"Missing Drosophila data in {DROSO_DATA_ROOT}: {names}"
        )


def ensure_chess_data():
    files = [
        (EVAL_DATA_ROOT / "puzzles.csv", f"{CHESS_DATA_URL}/puzzles.csv"),
        (EVAL_DATA_ROOT / "test" / "state_value_data.bag", f"{CHESS_DATA_URL}/test/state_value_data.bag"),
    ]
    for dest, url in files:
        if dest.exists():
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(url, dest)


def ensure_subsample(num_records, seed, exp_config):
    npy_path = subsample_npy_path(num_records, seed)
    if npy_path.exists():
        print(f"Subsample exists: {npy_path}")
        return npy_path

    bag_path = TRAIN_BAG_DIR / "state_value_data.bag"
    if not bag_path.exists():
        url = f"{CHESS_DATA_URL}/train/state_value_data.bag"
        print(f"Downloading train bag (~36 GB): {url}")
        urlretrieve(url, bag_path)

    policy_name = "state_value"
    transform_obj = _TRANSFORMATION_BY_POLICY[policy_name](copy.deepcopy(exp_config))
    dataset = ChessDataset(
        data_path=str(bag_path),
        coder_name=policy_name,
        transform_obj=transform_obj,
        num_records=num_records,
        seed=seed,
        sampling=True,
    )
    print(f"Creating subsample {num_records} (seed={seed})...")
    save_dataset_to_npy(dataset, str(npy_path), n_procs=SUBSAMPLE_NPROCS)
    return npy_path


def build_data_loader_notebook(config, exp_config):
    policy_name = "state_value"
    transform_obj = _TRANSFORMATION_BY_POLICY[policy_name](exp_config)
    generator = torch.Generator()
    generator.manual_seed(config.seed)

    if config.split == "train" and config.num_records is not None:
        npy_path = subsample_npy_path(config.num_records, config.seed)
        if npy_path.exists():
            dataset = NpyChessDataset(str(npy_path), transform_obj)
            return DataLoader(
                dataset,
                batch_size=config.batch_size,
                shuffle=config.shuffle,
                num_workers=20,
                persistent_workers=True,
                worker_init_fn=seed_worker,
                prefetch_factor=2,
                pin_memory=False,
                generator=generator,
            )
    return data_loader.build_data_loader(config=config, exp_config=exp_config)


def batch_size_for(n):
    return 32 if n <= 530 else 128


def train_one(train_num_sample, base_config, droso_config, exp_run_dir, device):
    exp_cfg = copy.deepcopy(base_config)
    exp_cfg["train_num_sample"] = train_num_sample
    exp_cfg["batch_size"] = batch_size_for(train_num_sample)
    exp_cfg["result_path"] = os.path.join(exp_run_dir, exp_cfg["result_path"])
    exp_cfg["device"] = device
    exp_cfg["droso_config"] = droso_config
    exp_cfg["chess_data_root"] = str(CHESS_DATA_ROOT)
    exp_cfg["eval_initial_puzzle"] = True
    exp_cfg["exp_id"] = get_exp_name(exp_cfg)

    out_path = get_out_path(exp_cfg)
    log_path = LOG_DIR / f"train_{train_num_sample}_seed{exp_cfg['seed']}.log"

    if SKIP_IF_EXISTS and os.path.exists(os.path.join(out_path, "model.pth")):
        return {"out_path": out_path, "skipped": True, "log_path": None}

    train_config = config_lib.TrainConfig(
        learning_rate=exp_cfg["learning_rate"],
        num_epoch=None,
        use_steps=exp_cfg["use_steps"],
        num_steps=exp_cfg["num_steps"],
        num_steps_eval=exp_cfg["num_steps_eval"],
        data=config_lib.DataConfig(
            model_choice=exp_cfg["model_choice"],
            batch_size=exp_cfg["batch_size"],
            shuffle=True,
            split="train",
            num_records=train_num_sample,
        ),
    )
    test_config = config_lib.EvalConfig(
        data=config_lib.DataConfig(
            model_choice=exp_cfg["model_choice"],
            batch_size=512,
            shuffle=False,
            split="test",
            num_records=530_310_443,
        ),
    )

    with log_to_file(log_path) as log_file:
        print(f"train_num_sample={train_num_sample}  batch_size={exp_cfg['batch_size']}")
        print(f"out_path={out_path}")
        ensure_subsample(train_num_sample, exp_cfg["seed"], exp_cfg)
        out_path = training.train(
            train_config=train_config,
            test_config=test_config,
            build_data_loader=build_data_loader_notebook,
            exp_config=exp_cfg,
        )
        plot_loss_curve(out_path)
        eval_DPU_on_puzzles(
            out_path,
            model_choice="best_test_model",
            save_name_suffix="best_test",
            chess_data_root=str(CHESS_DATA_ROOT),
        )
        print(f"Done. Results saved to {out_path}")

    return {"out_path": out_path, "skipped": False, "log_path": str(log_file)}


device = resolve_device(DEVICE)
CHESS_DATA_ROOT.mkdir(parents=True, exist_ok=True)
ensure_droso_data()
ensure_chess_data()

results = {}
for n in TRAIN_SIZES:
    results[n] = train_one(n, BASE_CONFIG, DROSO_CONFIG, EXP_RUN_DIR, device)

print(f"Device: {device}")
print(f"Logs: {LOG_DIR}")
for n, info in results.items():
    status = "skipped" if info["skipped"] else "done"
    print(f"  n={n} [{status}]  out={info['out_path']}")
    if info["log_path"]:
        print(f"         log={info['log_path']}")

## Plot loss & accuracy

Run after **Train**.
- **Plot**: train / test loss curves (`record.pkl`)
- **Text**: overall puzzle accuracy
- **Plot**: per-Elo-bucket puzzle accuracy bar chart

In [ ]:
import pickle

import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from src.evaluate.plotting import plot_bar

PUZZLE_PKL = "puzzle_result_best_test.pkl"


def load_record(out_path):
    pkl = Path(out_path) / "record.pkl"
    if not pkl.exists():
        print(f"Missing: {pkl}")
        return None
    with open(pkl, "rb") as f:
        return pickle.load(f)


def load_puzzle_results(out_path):
    pkl = Path(out_path) / PUZZLE_PKL
    if not pkl.exists():
        return None
    return pd.read_pickle(pkl)


# --- loss curves (train + test) ---
fig, axes = plt.subplots(1, len(results), figsize=(6 * max(len(results), 1), 4), squeeze=False)
plotted = False
for ax, (n, info) in zip(axes[0], results.items()):
    record = load_record(info["out_path"])
    if record is None:
        ax.set_title(f"n={n} (no record.pkl)")
        continue
    if "step_train_loss" in record:
        ax.plot(record["step_num"], record["step_test_loss"], label="Test")
        ax.plot(record["step_num"], record["step_train_loss"], label="Train")
        ax.set_xlabel("Step")
    else:
        epochs = range(1, len(record["epoch_test_loss"]) + 1)
        ax.plot(epochs, record["epoch_test_loss"], label="Test")
        ax.plot(epochs, record["epoch_train_loss"], label="Train")
        ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"train_num_sample={n}")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plotted = True

if not plotted:
    raise FileNotFoundError("No record.pkl found. Run Train first.")

plt.tight_layout()
plt.show()

# --- puzzle accuracy: overall (text) + Elo buckets (bar chart) ---
print("Puzzle accuracy (best_test model):")
fig, axes = plt.subplots(1, len(results), figsize=(8 * max(len(results), 1), 5), squeeze=False)
acc_plotted = False
for ax, (n, info) in zip(axes[0], results.items()):
    df = load_puzzle_results(info["out_path"])
    if df is None:
        print(f"  n={n}: missing {PUZZLE_PKL}")
        ax.set_title(f"n={n} (no {PUZZLE_PKL})")
        continue
    labels, pcts, overall = plot_bar(df, choice="elo", no_plot=True)
    print(f"  n={n}: overall = {overall:.4f}")
    x = range(len(labels))
    ax.bar(x, pcts)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_xlabel("Puzzle Rating (Elo) - Count")
    ax.set_title(f"train_num_sample={n}")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    acc_plotted = True

if acc_plotted:
    plt.tight_layout()
    plt.show()